In [1]:
# 01. IMPORT LIBRARIES

import pandas as pd
from pathlib import Path

In [2]:
# 02. SET PROJECT PATHS

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TCIR_METADATA = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "cleaned_satellite"
    / "tcir_metadata_cleaned.csv"
)

IBTRACS_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "ibtracs"
    / "ibtracs.since1980.list.v04r01.csv"
)

print("TCIR metadata:", TCIR_METADATA)
print("IBTrACS file:", IBTRACS_FILE)

TCIR metadata: e:\SIH\tropical-cyclone-ai\data\interim\cleaned_satellite\tcir_metadata_cleaned.csv
IBTrACS file: e:\SIH\tropical-cyclone-ai\data\raw\ibtracs\ibtracs.since1980.list.v04r01.csv


In [3]:
# 03. LOAD TCIR METADATA

tcir_df = pd.read_csv(TCIR_METADATA)

print("TCIR shape:", tcir_df.shape)
display(tcir_df.head())

TCIR shape: (10538, 8)


,REGION,CYCLONE_ID,LON,LAT,TIME,WIND_KTS,SIZE,PRESSURE_MB
0,ATLN,200301L,-66.0,31.4,2003-04-18 15:00:00,30.0,0.0,1008.0
1,ATLN,200301L,-66.3,31.9,2003-04-18 18:00:00,30.0,0.0,1007.0
2,ATLN,200301L,-66.6,32.5,2003-04-18 21:00:00,30.0,0.0,1007.0
3,ATLN,200301L,-68.6,34.5,2003-04-19 12:00:00,35.0,0.0,1006.0
4,ATLN,200301L,-68.8,34.4,2003-04-19 15:00:00,35.0,0.0,1006.0


In [4]:
# 04. LOAD GLOBAL IBTRACS

ibtracs_df = pd.read_csv(
    IBTRACS_FILE,
    low_memory=False,
    skiprows=[1]
)

print("Global IBTrACS shape:", ibtracs_df.shape)
display(ibtracs_df.head())

Global IBTrACS shape: (309202, 174)


,SID,SEASON,NUMBER,BASIN,SUBBASIN,NAME,ISO_TIME,NATURE,LAT,LON,...,BOM_GUST_PER,REUNION_GUST,REUNION_GUST_PER,USA_SEAHGT,USA_SEARAD_NE,USA_SEARAD_SE,USA_SEARAD_SW,USA_SEARAD_NW,STORM_SPEED,STORM_DIR
0,1980001S13173,1980,1,SP,MM,PENI,1980-01-01 00:00:00,TS,-12.5,172.5,...,,,,,,,,,6,350
1,1980001S13173,1980,1,SP,MM,PENI,1980-01-01 03:00:00,TS,-12.2,172.4,...,,,,,,,,,6,350
2,1980001S13173,1980,1,SP,MM,PENI,1980-01-01 06:00:00,TS,-11.9,172.4,...,,,,,,,,,5,360
3,1980001S13173,1980,1,SP,MM,PENI,1980-01-01 09:00:00,TS,-11.7,172.4,...,,,,,,,,,4,10
4,1980001S13173,1980,1,SP,MM,PENI,1980-01-01 12:00:00,TS,-11.5,172.5,...,,,,,,,,,4,20


In [5]:
# 05. INSPECT IMPORTANT IBTRACS COLUMNS

important_cols = [
    "SID",
    "SEASON",
    "BASIN",
    "ISO_TIME",
    "LAT",
    "LON",
    "USA_ATCF_ID",
    "USA_WIND",
    "USA_PRES"
]

available_cols = [
    col for col in important_cols
    if col in ibtracs_df.columns
]

print("Available columns:")
print(available_cols)

display(ibtracs_df[available_cols].head(10))

Available columns:
['SID', 'SEASON', 'BASIN', 'ISO_TIME', 'LAT', 'LON', 'USA_ATCF_ID', 'USA_WIND', 'USA_PRES']


,SID,SEASON,BASIN,ISO_TIME,LAT,LON,USA_ATCF_ID,USA_WIND,USA_PRES
0,1980001S13173,1980,SP,1980-01-01 00:00:00,-12.5,172.5,SH051980,25,
1,1980001S13173,1980,SP,1980-01-01 03:00:00,-12.2,172.4,SH051980,25,
2,1980001S13173,1980,SP,1980-01-01 06:00:00,-11.9,172.4,SH051980,25,
3,1980001S13173,1980,SP,1980-01-01 09:00:00,-11.7,172.4,SH051980,25,
4,1980001S13173,1980,SP,1980-01-01 12:00:00,-11.5,172.5,SH051980,25,
5,1980001S13173,1980,SP,1980-01-01 15:00:00,-11.3,172.6,SH051980,28,
6,1980001S13173,1980,SP,1980-01-01 18:00:00,-11.2,172.7,SH051980,30,
7,1980001S13173,1980,SP,1980-01-01 21:00:00,-11.2,172.9,SH051980,30,
8,1980001S13173,1980,SP,1980-01-02 00:00:00,-11.2,173.0,SH051980,30,
9,1980001S13173,1980,SP,1980-01-02 03:00:00,-11.3,173.1,SH051980,30,


In [6]:
# 06. CHECK CYCLONE ID MATCHING

tcir_ids = set(
    tcir_df["CYCLONE_ID"]
    .astype(str)
    .str.strip()
)

ibtracs_ids = set(
    ibtracs_df["USA_ATCF_ID"]
    .astype(str)
    .str.strip()
)

matched_ids = tcir_ids.intersection(ibtracs_ids)

print("TCIR unique cyclone IDs:", len(tcir_ids))
print("IBTrACS ATCF IDs:", len(ibtracs_ids))
print("Matched cyclone IDs:", len(matched_ids))
print(
    "Match percentage:",
    round(len(matched_ids) / len(tcir_ids) * 100, 2),
    "%"
)

TCIR unique cyclone IDs: 485
IBTrACS ATCF IDs: 4537
Matched cyclone IDs: 0
Match percentage: 0.0 %


In [7]:
# 07. INSPECT CYCLONE ID FIELDS

id_columns = [
    col for col in ibtracs_df.columns
    if "ID" in col.upper()
]

print("ID-related columns:")
print(id_columns)

print("\nSample values:")
for col in id_columns:
    print(f"\n{col}:")
    print(ibtracs_df[col].dropna().astype(str).head(10).tolist())

ID-related columns:
['SID', 'USA_ATCF_ID']

Sample values:

SID:
['1980001S13173', '1980001S13173', '1980001S13173', '1980001S13173', '1980001S13173', '1980001S13173', '1980001S13173', '1980001S13173', '1980001S13173', '1980001S13173']

USA_ATCF_ID:
['SH051980', 'SH051980', 'SH051980', 'SH051980', 'SH051980', 'SH051980', 'SH051980', 'SH051980', 'SH051980', 'SH051980']


In [8]:
# 08. CHECK TCIR ID IN ALL IBTRACS ID FIELDS

sample_tcir_ids = tcir_df["CYCLONE_ID"].astype(str).str.strip()

for col in id_columns:
    ib_ids = set(
        ibtracs_df[col]
        .dropna()
        .astype(str)
        .str.strip()
    )

    matches = set(sample_tcir_ids).intersection(ib_ids)

    print(
        f"{col}: {len(matches)} matching IDs"
    )

SID: 0 matching IDs
USA_ATCF_ID: 0 matching IDs


In [9]:
# 09. INSPECT TCIR CYCLONE IDS

display(
    tcir_df[
        ["REGION", "CYCLONE_ID", "TIME", "LAT", "LON"]
    ].head(20)
)

print("\nSample unique TCIR cyclone IDs:")
print(
    tcir_df["CYCLONE_ID"]
    .drop_duplicates()
    .head(30)
    .tolist()
)

,REGION,CYCLONE_ID,TIME,LAT,LON
0,ATLN,200301L,2003-04-18 15:00:00,31.4,-66.0
1,ATLN,200301L,2003-04-18 18:00:00,31.9,-66.3
2,ATLN,200301L,2003-04-18 21:00:00,32.5,-66.6
3,ATLN,200301L,2003-04-19 12:00:00,34.5,-68.6
4,ATLN,200301L,2003-04-19 15:00:00,34.4,-68.8
5,ATLN,200301L,2003-04-19 18:00:00,34.3,-69.1
6,ATLN,200301L,2003-04-19 21:00:00,34.0,-69.0
7,ATLN,200301L,2003-04-20 12:00:00,32.0,-68.2
8,ATLN,200301L,2003-04-20 15:00:00,31.9,-67.8
9,ATLN,200301L,2003-04-20 18:00:00,31.7,-67.3



Sample unique TCIR cyclone IDs:
['200301L', '200302L', '200303L', '200304L', '200305L', '200306L', '200307L', '200308L', '200309L', '200310L', '200311L', '200312L', '200313L', '200314L', '200315L', '200316L', '200317L', '200318L', '200319L', '200320L', '200321L', '200401L', '200402L', '200403L', '200404L', '200405L', '200406L', '200407L', '200408L', '200409L']


In [10]:
# 10. CHECK TCIR CYCLONE ID PATTERN

id_summary = (
    tcir_df
    .groupby(["REGION", "CYCLONE_ID"])
    .agg(
        START_TIME=("TIME", "min"),
        END_TIME=("TIME", "max"),
        OBSERVATIONS=("TIME", "count")
    )
    .reset_index()
)

display(id_summary.head(20))

,REGION,CYCLONE_ID,START_TIME,END_TIME,OBSERVATIONS
0,ATLN,200301L,2003-04-18 15:00:00,2003-04-27 12:00:00,40
1,ATLN,200302L,2003-06-11 09:00:00,2003-06-11 18:00:00,4
2,ATLN,200303L,2003-06-28 12:00:00,2003-07-03 00:00:00,25
3,ATLN,200304L,2003-07-07 12:00:00,2003-07-17 00:00:00,46
4,ATLN,200305L,2003-07-16 12:00:00,2003-07-26 21:00:00,54
5,ATLN,200306L,2003-07-19 18:00:00,2003-07-21 12:00:00,8
6,ATLN,200307L,2003-07-25 12:00:00,2003-07-27 00:00:00,10
7,ATLN,200308L,2003-08-15 12:00:00,2003-08-16 21:00:00,6
8,ATLN,200309L,2003-08-22 12:00:00,2003-08-22 12:00:00,1
9,ATLN,200310L,2003-08-27 18:00:00,2003-09-08 18:00:00,51


In [11]:
# 11. COMPARE TCIR AND IBTRACS BY YEAR AND BASIN

tcir_sample = (
    tcir_df[["REGION", "CYCLONE_ID"]]
    .drop_duplicates()
    .head(20)
)

display(tcir_sample)

,REGION,CYCLONE_ID
0,ATLN,200301L
40,ATLN,200302L
44,ATLN,200303L
69,ATLN,200304L
115,ATLN,200305L
169,ATLN,200306L
177,ATLN,200307L
187,ATLN,200308L
193,ATLN,200309L
194,ATLN,200310L


In [12]:
# 12. INSPECT 2003 IBTRACS CYCLONES

ib_2003 = ibtracs_df[
    ibtracs_df["SEASON"] == 2003
][
    ["SID", "SEASON", "BASIN", "ISO_TIME", "LAT", "LON", "USA_ATCF_ID"]
].copy()

display(
    ib_2003.drop_duplicates("SID").head(50)
)

,SID,SEASON,BASIN,ISO_TIME,LAT,LON,USA_ATCF_ID
159311,2002247S03067,2003,SI,2002-09-04 06:00:00,-2.5,67.2,
160642,2002307S07070,2003,SI,2002-11-03 06:00:00,-6.6,69.6,SH022003
160764,2002319S06078,2003,SI,2002-11-14 18:00:00,-6.3,77.9,SH032003
161058,2002335S13182,2003,SP,2002-12-01 06:00:00,-13.2,-178.3,SH042003
161175,2002356S07070,2003,SI,2002-12-21 18:00:00,-6.8,69.5,
161274,2002358S08185,2003,SP,2002-12-23 18:00:00,-8.0,184.5,
161369,2002359S03089,2003,SI,2002-12-25 00:00:00,-2.8,88.5,SH072003
161444,2002364S16045,2003,SI,2002-12-30 06:00:00,-15.8,44.8,
161565,2003004S10136,2003,SP,2003-01-04 00:00:00,-10.0,135.5,
161739,2003007S10072,2003,SI,2003-01-06 12:00:00,-9.5,71.8,SH092003


In [17]:
# 13. INSPECT 2003 ATLANTIC STORMS

atlantic_2003 = ibtracs_df[
    (ibtracs_df["SEASON"] == 2003) &
    (ibtracs_df["USA_ATCF_ID"].astype(str).str.startswith("AL"))
][
    ["SID", "SEASON", "BASIN", "ISO_TIME", "LAT", "LON", "USA_ATCF_ID"]
].copy()

display(
    atlantic_2003
    .drop_duplicates("SID")
    .head(30)
)

,SID,SEASON,BASIN,ISO_TIME,LAT,LON,USA_ATCF_ID
163474,2003108N29294,2003,NaN,2003-04-18 00:00:00,29.1,-66.2,AL012003
164264,2003162N10319,2003,NaN,2003-06-11 00:00:00,9.5,-40.8,AL022003
164355,2003179N20271,2003,NaN,2003-06-28 06:00:00,19.5,-89.0,AL032003
164416,2003188N11307,2003,NaN,2003-07-07 00:00:00,11.1,-53.5,AL042003
164700,2003198N31306,2003,NaN,2003-07-16 12:00:00,30.8,-54.1,AL052003
164842,2003201N12317,2003,NaN,2003-07-19 18:00:00,12.3,-43.5,AL062003
164882,2003207N29280,2003,NaN,2003-07-25 12:00:00,29.3,-80.1,AL072003
165267,2003227N26277,2003,NaN,2003-08-14 18:00:00,26.4,-83.3,AL082003
165323,2003234N15295,2003,NaN,2003-08-21 18:00:00,14.5,-65.5,AL092003
165442,2003240N15329,2003,NaN,2003-08-27 18:00:00,14.6,-30.7,AL102003


In [18]:
# 14. CONVERT TCIR ID TO IBTRACS ATCF ID

def convert_tcir_id(tcir_id):
    tcir_id = str(tcir_id)

    year = tcir_id[:4]
    storm_number = tcir_id[4:6]
    region_code = tcir_id[6]

    region_map = {
        "L": "AL",
        "E": "EP",
        "W": "WP"
    }

    atcf_prefix = region_map.get(region_code)

    if atcf_prefix is None:
        return None

    return f"{atcf_prefix}{storm_number}{year}"


tcir_df["USA_ATCF_ID"] = (
    tcir_df["CYCLONE_ID"]
    .apply(convert_tcir_id)
)

display(
    tcir_df[
        ["REGION", "CYCLONE_ID", "USA_ATCF_ID"]
    ].drop_duplicates().head(20)
)

,REGION,CYCLONE_ID,USA_ATCF_ID
0,ATLN,200301L,AL012003
40,ATLN,200302L,AL022003
44,ATLN,200303L,AL032003
69,ATLN,200304L,AL042003
115,ATLN,200305L,AL052003
169,ATLN,200306L,AL062003
177,ATLN,200307L,AL072003
187,ATLN,200308L,AL082003
193,ATLN,200309L,AL092003
194,ATLN,200310L,AL102003


In [19]:
# 15. VERIFY ATCF ID MATCHING

tcir_atcf_ids = set(
    tcir_df["USA_ATCF_ID"]
    .dropna()
    .astype(str)
)

ibtracs_atcf_ids = set(
    ibtracs_df["USA_ATCF_ID"]
    .dropna()
    .astype(str)
)

matched_ids = tcir_atcf_ids.intersection(
    ibtracs_atcf_ids
)

print("TCIR ATCF IDs:", len(tcir_atcf_ids))
print("IBTrACS ATCF IDs:", len(ibtracs_atcf_ids))
print("Matched IDs:", len(matched_ids))

print(
    "Match percentage:",
    round(
        len(matched_ids) / len(tcir_atcf_ids) * 100,
        2
    ),
    "%"
)

TCIR ATCF IDs: 485
IBTrACS ATCF IDs: 4537
Matched IDs: 485
Match percentage: 100.0 %


In [20]:
# 16. CHECK UNMATCHED IDS

unmatched_ids = sorted(
    tcir_atcf_ids - ibtracs_atcf_ids
)

print("Unmatched TCIR IDs:", len(unmatched_ids))
print(unmatched_ids[:30])

Unmatched TCIR IDs: 0
[]


In [21]:
# 17. PREPARE IBTRACS FOR FUSION

ib_fusion = ibtracs_df[
    [
        "USA_ATCF_ID",
        "ISO_TIME",
        "SID",
        "LAT",
        "LON",
        "USA_WIND",
        "USA_PRES"
    ]
].copy()

ib_fusion["ISO_TIME"] = pd.to_datetime(
    ib_fusion["ISO_TIME"],
    errors="coerce"
)

ib_fusion["USA_ATCF_ID"] = (
    ib_fusion["USA_ATCF_ID"]
    .astype(str)
    .str.strip()
)

print("IBTrACS fusion rows:", len(ib_fusion))

IBTrACS fusion rows: 309202


In [28]:
# 18. CHECK IBTRACS FUSION KEYS

tcir_atcf_ids = set(
    tcir_df["USA_ATCF_ID"]
    .dropna()
    .astype(str)
    .str.strip()
)

ib_fusion_matched = ib_fusion[
    ib_fusion["USA_ATCF_ID"].isin(tcir_atcf_ids)
].copy()

duplicate_keys = ib_fusion_matched.duplicated(
    subset=["USA_ATCF_ID", "ISO_TIME"],
    keep=False
)

print("IBTrACS rows for TCIR cyclones:", len(ib_fusion_matched))
print("Duplicate ID + time records:", duplicate_keys.sum())

if duplicate_keys.sum() > 0:
    display(
        ib_fusion_matched[duplicate_keys]
        .sort_values(["USA_ATCF_ID", "ISO_TIME"])
        .head(20)
    )

IBTrACS rows for TCIR cyclones: 28617
Duplicate ID + time records: 0


In [29]:
# 19. LOAD ORIGINAL TCIR LABELS

import numpy as np

LABEL_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "satellite"
    / "tcir"
    / "Cyclone_Labels h5.npy"
)

labels_full = np.load(
    LABEL_FILE,
    allow_pickle=True
)

tcir_full = pd.DataFrame(
    labels_full,
    columns=[
        "REGION",
        "CYCLONE_ID",
        "LON",
        "LAT",
        "TIME",
        "WIND_KTS",
        "SIZE",
        "PRESSURE_MB"
    ]
)

print("Original TCIR rows:", len(tcir_full))

Original TCIR rows: 21076


In [30]:
# 20. PREPARE TCIR METADATA

tcir_full["LON"] = pd.to_numeric(
    tcir_full["LON"],
    errors="coerce"
)

tcir_full["LAT"] = pd.to_numeric(
    tcir_full["LAT"],
    errors="coerce"
)

tcir_full["WIND_KTS"] = pd.to_numeric(
    tcir_full["WIND_KTS"],
    errors="coerce"
)

tcir_full["SIZE"] = pd.to_numeric(
    tcir_full["SIZE"],
    errors="coerce"
)

tcir_full["PRESSURE_MB"] = pd.to_numeric(
    tcir_full["PRESSURE_MB"],
    errors="coerce"
)

tcir_full["TIME"] = pd.to_datetime(
    tcir_full["TIME"].astype(str),
    format="%Y%m%d%H",
    errors="coerce"
)

tcir_full["USA_ATCF_ID"] = (
    tcir_full["CYCLONE_ID"]
    .apply(convert_tcir_id)
)

print("TCIR rows:", len(tcir_full))
print("Missing values:")
display(tcir_full.isna().sum())

TCIR rows: 21076
Missing values:


REGION         0
CYCLONE_ID     0
LON            0
LAT            0
TIME           0
WIND_KTS       0
SIZE           0
PRESSURE_MB    0
USA_ATCF_ID    0
dtype: int64

In [32]:
# 21. FUSE TCIR WITH IBTRACS

fused_df = tcir_full.merge(
    ib_fusion,
    left_on=["USA_ATCF_ID", "TIME"],
    right_on=["USA_ATCF_ID", "ISO_TIME"],
    how="left",
    suffixes=("_TCIR", "_IBTRACS")
)

print("TCIR rows:", len(tcir_full))
print("Fused rows:", len(fused_df))

print(
    "IBTrACS matches:",
    fused_df["SID"].notna().sum()
)

print(
    "Unmatched rows:",
    fused_df["SID"].isna().sum()
)

TCIR rows: 21076
Fused rows: 21076
IBTrACS matches: 21076
Unmatched rows: 0


In [34]:
# 22. VALIDATE LOCATIONS

print("Fused columns:")
print(fused_df.columns.tolist())

Fused columns:
['REGION', 'CYCLONE_ID', 'LON_TCIR', 'LAT_TCIR', 'TIME', 'WIND_KTS', 'SIZE', 'PRESSURE_MB', 'USA_ATCF_ID', 'ISO_TIME', 'SID', 'LAT_IBTRACS', 'LON_IBTRACS', 'USA_WIND', 'USA_PRES']


In [39]:
# 23. VALIDATE AND NORMALIZE LOCATIONS

# Normalize longitude to -180 to +180
fused_df["LON_TCIR_NORM"] = (
    (fused_df["LON_TCIR"] + 180) % 360
) - 180

fused_df["LON_IBTRACS_NORM"] = (
    (fused_df["LON_IBTRACS"] + 180) % 360
) - 180

# Calculate differences
fused_df["LAT_DIFF"] = (
    fused_df["LAT_TCIR"] - fused_df["LAT_IBTRACS"]
)

fused_df["LON_DIFF"] = (
    fused_df["LON_TCIR_NORM"] -
    fused_df["LON_IBTRACS_NORM"]
)

print("Latitude difference:")
display(fused_df["LAT_DIFF"].describe())

print("\nLongitude difference:")
display(fused_df["LON_DIFF"].describe())

Latitude difference:


count    21076.000000
mean         0.011596
std          0.115865
min         -2.300000
25%          0.000000
50%          0.000000
75%          0.000000
max          2.000000
Name: LAT_DIFF, dtype: float64


Longitude difference:


count    21076.000000
mean        -0.064623
std          4.923523
min       -359.700000
25%          0.000000
50%          0.000000
75%          0.000000
max          3.200000
Name: LON_DIFF, dtype: float64

In [40]:
# 24. PREVIEW FUSED DATA

display(
    fused_df[
        [
            "REGION",
            "CYCLONE_ID",
            "USA_ATCF_ID",
            "TIME",
            "LAT_TCIR",
            "LON_TCIR",
            "LAT_IBTRACS",
            "LON_IBTRACS",
            "LAT_DIFF",
            "LON_DIFF",
            "WIND_KTS",
            "PRESSURE_MB",
            "USA_WIND",
            "USA_PRES"
        ]
    ].head(20)
)

,REGION,CYCLONE_ID,USA_ATCF_ID,TIME,LAT_TCIR,LON_TCIR,LAT_IBTRACS,LON_IBTRACS,LAT_DIFF,LON_DIFF,WIND_KTS,PRESSURE_MB,USA_WIND,USA_PRES
0,ATLN,200301L,AL012003,2003-04-18 15:00:00,31.4,-66.0,31.3,-66.0,0.1,0.0,30.0,1008.0,30,1008
1,ATLN,200301L,AL012003,2003-04-18 18:00:00,31.9,-66.3,31.9,-66.3,0.0,0.0,30.0,1007.0,30,1007
2,ATLN,200301L,AL012003,2003-04-18 21:00:00,32.5,-66.6,32.5,-66.6,0.0,0.0,30.0,1007.0,30,1007
3,ATLN,200301L,AL012003,2003-04-19 12:00:00,34.5,-68.6,34.5,-68.6,0.0,0.0,35.0,1006.0,35,1006
4,ATLN,200301L,AL012003,2003-04-19 15:00:00,34.4,-68.8,34.5,-68.9,-0.1,0.1,35.0,1006.0,35,1006
5,ATLN,200301L,AL012003,2003-04-19 18:00:00,34.3,-69.1,34.3,-69.1,0.0,0.0,35.0,1006.0,35,1006
6,ATLN,200301L,AL012003,2003-04-19 21:00:00,34.0,-69.0,34.0,-69.1,0.0,0.1,38.0,1006.0,38,1006
7,ATLN,200301L,AL012003,2003-04-20 12:00:00,32.0,-68.2,32.0,-68.2,0.0,0.0,45.0,1000.0,45,1000
8,ATLN,200301L,AL012003,2003-04-20 15:00:00,31.9,-67.8,31.8,-67.8,0.1,0.0,45.0,999.0,45,999
9,ATLN,200301L,AL012003,2003-04-20 18:00:00,31.7,-67.3,31.7,-67.3,0.0,0.0,45.0,998.0,45,998


In [41]:
# 25. SAVE FUSED DATASET

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "fused_dataset"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FUSED_FILE = OUTPUT_DIR / "tcir_ibtracs_fused.csv"

fused_df.to_csv(
    FUSED_FILE,
    index=False
)

print("Saved:", FUSED_FILE)
print("Final shape:", fused_df.shape)

Saved: e:\SIH\tropical-cyclone-ai\data\processed\fused_dataset\tcir_ibtracs_fused.csv
Final shape: (21076, 19)


In [42]:
# 26. PREPARE FINAL FUSED DATASET

final_fused_df = fused_df.drop(
    columns=[
        "LON_TCIR_NORM",
        "LON_IBTRACS_NORM",
        "LAT_DIFF",
        "LON_DIFF"
    ],
    errors="ignore"
).copy()

print("Final shape:", final_fused_df.shape)
print("Final columns:")
print(final_fused_df.columns.tolist())

Final shape: (21076, 15)
Final columns:
['REGION', 'CYCLONE_ID', 'LON_TCIR', 'LAT_TCIR', 'TIME', 'WIND_KTS', 'SIZE', 'PRESSURE_MB', 'USA_ATCF_ID', 'ISO_TIME', 'SID', 'LAT_IBTRACS', 'LON_IBTRACS', 'USA_WIND', 'USA_PRES']


In [43]:
# 27. SAVE FINAL FUSED DATASET

final_fused_df.to_csv(
    FUSED_FILE,
    index=False
)

print("Saved:", FUSED_FILE)
print("Final shape:", final_fused_df.shape)

Saved: e:\SIH\tropical-cyclone-ai\data\processed\fused_dataset\tcir_ibtracs_fused.csv
Final shape: (21076, 15)


In [44]:
# 28. FINAL FUSION VERIFICATION

print("Final fused rows:", len(final_fused_df))
print("Final fused columns:", len(final_fused_df.columns))
print("Unmatched IBTrACS:", final_fused_df["SID"].isna().sum())
print("Missing values:", final_fused_df.isna().sum().sum())
print("Unique cyclones:", final_fused_df["CYCLONE_ID"].nunique())

Final fused rows: 21076
Final fused columns: 15
Unmatched IBTrACS: 0
Missing values: 0
Unique cyclones: 485


## 29. DATA FUSION SUMMARY

- TCIR images: 21,076
- TCIR metadata rows: 21,076
- IBTrACS matches: 21,076
- Unmatched records: 0
- Unique cyclones: 485
- Final fused dataset: `data/processed/fused_dataset/tcir_ibtracs_fused.csv`
- Final dataset shape: 21,076 rows × 15 columns

The TCIR image metadata was successfully matched with global IBTrACS records using cyclone ID and timestamp. The final fused dataset is ready for downstream model development.